# Breast Cancer

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import torch

import SpaDiff as sd
from SpaDiff.utils import cal_purity, set_seed

## Configuration

In [ ]:
SEED = 42
SLICE_ORDER = ["H_1", "H_2", "H_3"]
TRUTH_KEY = "pathologist"

DATA_ROOT = Path("path/breastcancer/")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)

## Data loading

In [ ]:
samples = {}
for sample in SLICE_ORDER:
    current = sc.read_h5ad(DATA_ROOT / f"{sample}.h5ad")
    current.var_names_make_unique()
    current.layers["counts"] = current.X.copy()
    sc.pp.normalize_total(current, target_sum=1e4)
    sc.pp.log1p(current)
    sc.pp.highly_variable_genes(current, flavor="seurat_v3", layer="counts", n_top_genes=3000)
    # current, _ = sd.spatial_reconstruction( current, alpha=1.0, n_neighbors=6)
    samples[sample] = current

hvg_union = set().union(*(set(current.var_names[current.var["highly_variable"]]) for current in samples.values()))
adata = sc.concat(samples, join="inner", label="batch_name", index_unique="-")
adata = adata[:, adata.var_names.isin(hvg_union)].copy()
adata.obs["batch_name"] = pd.Categorical(adata.obs["batch_name"], categories=SLICE_ORDER, ordered=True)
print(adata)
print(adata.obs["batch_name"].value_counts())

In [ ]:
# sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer="counts", n_top_genes=3000, batch_key="batch_name", subset=True)
sc.tl.pca(adata, n_comps=50)
pca_array = np.ascontiguousarray(adata.obsm["X_pca"], dtype=np.float32)
features = torch.from_numpy(pca_array).to(device)

## Simplicial complex

In [ ]:
topology = sd.build_spatial_topology(
    adata,
    slice_order=SLICE_ORDER,
    device=device,
)
operators = topology.operators

## Conditional VP-SDE training

In [ ]:
config = sd.SpaDiffConfig(
    num_batches=len(SLICE_ORDER),

)
model = sd.SpaDiff(config).to(device)
adata = model.fit_transform(
    adata,
    features,
    operators,
    batch_order=SLICE_ORDER,
    ode_steps=250,
)

In [ ]:
adata = sd.write_denoised_expression(adata, latent_key="X_spadiff", adjacency=topology.adjacency)
print("denoised layer:", adata.layers["spadiff_denoised"].shape)

## Louvain spatial domains

In [ ]:
sc.pp.neighbors(adata, use_rep='spadiff', n_neighbors=15, random_state=SEED)
sc.tl.louvain( adata, key_added="louvain", resolution=0.91, random_state=SEED)

In [ ]:
for sample in SLICE_ORDER:
    subset_obs = adata.obs.loc[adata.obs["batch_name"] == sample]

    valid = subset_obs[[TRUTH_KEY, "louvain"]].dropna()
    truth_codes = pd.Categorical(valid[TRUTH_KEY]).codes
    pred_codes = pd.Categorical(valid["louvain"]).codes
    purity = cal_purity(truth_codes, pred_codes)
    print(f"{sample}: purity={purity:.3f}")

valid = adata.obs[[TRUTH_KEY, "louvain"]].dropna()
truth_codes = pd.Categorical(valid[TRUTH_KEY]).codes
pred_codes = pd.Categorical(valid["louvain"]).codes
overall_purity = cal_purity(truth_codes, pred_codes)
print(f"All purity={overall_purity:.3f}")

In [ ]:
plot_color = ["#7495D3", "#59BE86", "#FEB915", "#C798EE", "#6D1A9C", "#F56867", "#D1D1D1"]
_, axes = plt.subplots(1, len(SLICE_ORDER), figsize=(18, 5))
for axis, sample in zip(np.atleast_1d(axes), SLICE_ORDER):
    subset = adata[adata.obs["batch_name"] == sample].copy()
    sc.pl.spatial(
        subset, color="louvain", ax=axis, show=False, spot_size=280,
        palette=plot_color, title=sample,
        # legend_loc=None,
    )
plt.tight_layout()
plt.show()

In [ ]:
sc.tl.umap(adata, random_state=SEED)
colors = ["batch_name", "louvain"]
if TRUTH_KEY in adata.obs:
    colors.append(TRUTH_KEY)
sc.pl.umap(
    adata, color=colors, size=60,
    legend_fontsize=11, legend_fontoutline=2,
)

In [ ]:
gene = "CD24"
layer = "spadiff_denoised"

fig, axes = plt.subplots( 1, len(SLICE_ORDER), figsize=(5 * len(SLICE_ORDER), 5))

for ax, sample in zip(axes, SLICE_ORDER):

    adata_slice = adata[adata.obs["batch_name"].astype(str) == str(sample)].copy()

    sc.pl.spatial(
        adata_slice,
        color=gene,
        layer=layer,
        spot_size=280,
        color_map="inferno",
        legend_loc=None,
        frameon=False,
        title=str(sample),
        ax=ax,
        show=False,
    )

plt.tight_layout()
plt.show()